# **Notebook 4: RAG Implementation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [1]:
!pip install -U ragas langchain langchain-core langchain-community langchain-huggingface langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 76.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.6/145.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 93.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.3 MB/s eta 0:00:00:00:0100:01


In [4]:
# YOUR CODE HERE

import os
import glob
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


documents = []

for filepath in glob.glob("/kaggle/input/datasets/piyushshx/sop-documents/*.md"):
    loader = TextLoader(filepath, encoding="utf-8")
    docs = loader.load()
    for doc in docs:
        doc.metadata["filename"] = os.path.basename(filepath)
    documents.extend(docs)

print(f"Total documents loaded: {len(documents)}")
print(documents[0].metadata)
print(documents[0].page_content[:200])

embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
# Validate embedding dimensions
sample_embedding = embedding_function.embed_query("test query")
print(f"Embedding dimensions: {len(sample_embedding)}")  # should print 384

Total documents loaded: 13
{'source': '/kaggle/input/datasets/piyushshx/sop-documents/account_recovery.md', 'filename': 'account_recovery.md'}
# Account Recovery

## When To Use
Use this procedure when a customer cannot access their account due to a
lockout, a lost email address, suspected unauthorized access, or a disabled
account — anythin


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimensions: 384


#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [5]:
# YOUR CODE HERE

import os
import shutil

persist_directory = "/kaggle/working/chroma_db"

if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)

vectordb = Chroma.from_documents(
    documents=documents,
    embedding=embedding_function,
    persist_directory=persist_directory
)

print("Documents loaded:", len(documents))
print("Records in index:", vectordb._collection.count())

assert vectordb._collection.count() == len(documents)

Documents loaded: 13
Records in index: 13


#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [6]:

import json
import re
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit quantization configuration (NF4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Load the model in 4-bit
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model.eval()

print(f"Loaded: {MODEL_NAME}")
print(f"Model device: {model.device}")
print(f"4-bit quantization enabled: {model.is_loaded_in_4bit}")

assert model.is_loaded_in_4bit, "Model is not loaded in 4-bit!"

# Keep the existing response parser
def parse_response(text):
    """Strip code fences, parse JSON. Returns (parsed_or_None, is_valid)."""
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned).strip()
    try:
        return json.loads(cleaned), True
    except json.JSONDecodeError:
        return None, False

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct
Model device: cuda:0
4-bit quantization enabled: True


In [7]:
# YOUR CODE HERE

def run_rag_inference(query, k=1):
    # Step 1: Retrieve top-k SOP documents
    retrieved_docs = vectordb.similarity_search(query, k=k)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    retrieved_source = [doc.metadata['filename'] for doc in retrieved_docs]
    
    # Step 2: Augment prompt with retrieved context
    messages = [
        {
            "role": "system",
            "content": f"Answer strictly using this SOP: {context}\n\nExtract the intent and category and return a valid JSON object only. No explanation."
        },
        {
            "role": "user",
            "content": query
        }
    ]
    
    # Step 3: Run inference
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            temperature=None,
            top_p=None
        )
    
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    
    return response, retrieved_source


# Test with ambiguous query
test_queries = [
    "I ordered something 2 weeks ago and it still hasn't arrived!",
    "I need a refund for my order",
    "How do I reset my password?"
]

for query in test_queries:
    response, sources = run_rag_inference(query, k=1)
    parsed, is_valid = parse_response(response)
    print(f"Query:     {query[:60]}")
    print(f"Retrieved: {sources}")
    print(f"Response:  {response}")
    print(f"Valid JSON: {is_valid}")
    print("-" * 60)

Query:     I ordered something 2 weeks ago and it still hasn't arrived!
Retrieved: ['shipping_delays.md']
Response:  {
  "intent": "customer_service",
  "category": "shipping_delays"
}
Valid JSON: True
------------------------------------------------------------
Query:     I need a refund for my order
Retrieved: ['refund_policy.md']
Response:  ```json
{
  "intent": "refund",
  "category": "customer service"
}
```
Valid JSON: True
------------------------------------------------------------
Query:     How do I reset my password?
Retrieved: ['password_reset.md']
Response:  {
  "intent": "password reset",
  "category": "self-service"
}
Valid JSON: True
------------------------------------------------------------


## Naive RAG Findings

**Retrieval Accuracy: 3/3 (100%)** — correct SOP retrieved for all test queries

**Remaining Issues:**
- Intent labels still generic — not aligned with Bitext schema
- Category values inconsistent casing and naming
- Model still relies on post-processing to fix code fences

**Conclusion:** Retrieval works well for clean queries.
Fine-tuning (Stage 4) is needed to fix intent precision and eliminate code fences.

---
## Save Artifacts for Downstream Notebooks

In [8]:
# YOUR CODE HERE
import json, os
from tqdm.auto import tqdm

IN_PATH = "/kaggle/input/datasets/piyushshx/artifacts-2/outputs.json"
OUT_DIR = "/kaggle/working/artifacts"
OUT_PATH = f"{OUT_DIR}/outputs.json"
os.makedirs(OUT_DIR, exist_ok=True)

BATCH_SIZE = 8        # lower to 4 if you hit CUDA out-of-memory, raise to 16 if memory is free
MAX_NEW_TOKENS = 64   # the JSON is ~30 tokens; 120 is more than needed
LIMIT = None          # set to 10 first to time a small run, then None for everything

with open(IN_PATH) as f:
    outputs = json.load(f)
if LIMIT:
    outputs = outputs[:LIMIT]

tokenizer.padding_side = "left"          # required for batched generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_prompt(query, k=1):
    docs = vectordb.similarity_search(query, k=k)
    context = "\n\n".join(d.page_content for d in docs)
    sources = [d.metadata["filename"] for d in docs]
    messages = [
        {"role": "system",
         "content": f"Answer strictly using this SOP: {context}\n\nExtract the intent and category and return a valid JSON object only. No explanation."},
        {"role": "user", "content": query},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return text, sources

# 1) Retrieval for all records (fast)
prompts, sources_list = [], []
for r in tqdm(outputs, desc="Retrieving"):
    text, src = build_prompt(r["test_query"], k=1)
    prompts.append(text)
    sources_list.append(src)

# 2) Sort by prompt length so each batch has little padding
order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
responses = [None] * len(prompts)

# 3) Batched generation
for start in tqdm(range(0, len(order), BATCH_SIZE), desc="Generating"):
    idx = order[start:start + BATCH_SIZE]
    batch = tokenizer([prompts[i] for i in idx], return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **batch,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen = out[:, batch["input_ids"].shape[1]:]
    for i, g in zip(idx, gen):
        responses[i] = tokenizer.decode(g, skip_special_tokens=True)

# 4) Write results back
for r, resp, src in zip(outputs, responses, sources_list):
    _, is_valid = parse_response(resp)
    r["naive_rag_output"] = resp
    r["rag_retrieved_source"] = src
    r["rag_valid_json"] = is_valid

with open(OUT_PATH, "w") as f:
    json.dump(outputs, f, indent=2)

print(f"outputs.json updated with naive_rag_output ✅  ({len(outputs)} records)")

Retrieving:   0%|          | 0/391 [00:00<?, ?it/s]

Generating:   0%|          | 0/49 [00:00<?, ?it/s]

outputs.json updated with naive_rag_output ✅  (391 records)


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [ ] SOP documents loaded via `TextLoader`
- [ ] **Embeddings generated and validated** ← _Task 3.2.1_
- [ ] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [ ] Naive similarity search executed on `test_query`
- [ ] Naive RAG output generated with SOP context injected
- [ ] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [ ] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**